# <span style="color:#57a3f5;"> **Activity Analysis and Calibration of HPGe Spectra**

The goal of this notebook is to generate activity curves for each of the five experimental jobs and analyze how the initial activity ($A_0$) relates to the irradiation time.

<br>

---

#### **Plan**

* Step 1: *Load and calibrate the detector efficiency* using well-known calibration sources (Ba-133, Cs-137, Eu-152).

* Step 2: *Load the experimental spectra* from the different jobs and apply the efficiency calibration.

* Step 3: *Calculate the activity for each job* based on the calibrated spectra and irradiation times.
  
* Step 4: *Visualize and analyze the results* by plotting activity curves and comparing $A_0$ vs. irradiation time.  

<br>

---

<br><br>

### <span style="color:#57a3f5;"> **Step 1: Efficiency Calibration with Calibration Sources**

We start by loading calibration sources and performing detector efficiency calibration, ensuring accurate conversion from counts to activity for our spectra.

Loading spectra for Ba-133, Cs-137, and Eu-152, setting isotopes, auto-calibrating energy for Eu-152 (due to its complex peak structure causing calibration errors), and performing efficiency calibration using their known activities and reference dates:

In [2]:
from pathlib import Path
import pandas as pd
import curie as ci

cal_path = Path("spectra/calibration")
CI_TO_BQ = 3.7e10  # Conversion constant (1 Ci = 3.7e10 Bq)

isotopes = ["133BA", "137CS", "152EU"]
prefixes = ["AB", "AA", "AC"]

spectra = []
for iso, prefix in zip(isotopes, prefixes):
    filename = f"{prefix}110625_{iso}.Spe"
    sp = ci.Spectrum(str(cal_path / filename))
    sp.isotopes = [iso]
    if iso == "152EU":
        sp.auto_calibrate()
    spectra.append(sp)


sources = pd.DataFrame([
    {"isotope": "133BA", "A0": 10.78e-6 * CI_TO_BQ, "ref_date": "10/01/1988 12:00:00"},
    {"isotope": "137CS", "A0": 11.46e-6 * CI_TO_BQ, "ref_date": "02/01/1978 12:00:00"},
    {"isotope": "152EU", "A0": 1.5e5, "ref_date": "01/01/2002 12:00:00"},
])


cb = ci.Calibration()
cb.calibrate(spectra, sources)
cb.plot_effcal(saveas="figures/efficiency_calibration.pdf", show=False)
cb.saveas("results/efficiency_calibration.json")

print(f"{'-'*32}\nEfficiency calibration complete!\n{'-'*32}")

--------------------------------
Efficiency calibration complete!
--------------------------------


<br><br>

### <span style="color:#57a3f5;"> **Step 2: Load the experimental spectra from the different jobs and apply the efficiency calibration**

Loading and preparing silver (Ag) spectra by reading files matching "job*.Spe", setting the isotopes of interest (Ag-108 and Ag-110), applying an existing efficiency calibration, and storing the spectra for further analysis:

In [2]:
exp_path = Path("spectra/experiment")
spectra_files = sorted(exp_path.glob("job*.Spe"))

cb = ci.Calibration("results/efficiency_calibration.json")
exp_spectra = []
for file in spectra_files:
    sp = ci.Spectrum(str(file))
    sp.isotopes = ["108AG", "110AG"]
    sp.fit_config = {"SNR_min": 3.5}
    sp.cb = cb
    exp_spectra.append(sp)

<br><br>

### <span style="color:#57a3f5;"> **Step 3: Calculate the activity for each job based on the calibrated spectra and irradiation times**

Text...

In [1]:
from activation_analysis import ActivityAnalyzer


delay_times = [14, 11, 15, 11, 7]
for i, delay_time in enumerate(delay_times):
    analyzer = ActivityAnalyzer(
            spectra_dir="spectra/experiment",
            calibration_path="results/efficiency_calibration.json",
            isotopes=["108AG", "110AG"],
            job_filter=f"job{i+1}" 
        )
    analyzer.load_calibration()
    analyzer.load_spectra()
    analyzer.analyze(delay_time=delay_time)
    analyzer.plot_activities()

A0 = 1.05e+04 ± 2.02e+03 Bq, R² = -1.465
A0 = 9.63e+04 ± 9.12e+03 Bq, R² = 0.984
A0 = 2.87e+04 ± 7.57e+03 Bq, R² = 0.864
A0 = 1.20e+05 ± 9.27e+03 Bq, R² = 0.997
A0 = 4.44e+04 ± 3.43e+03 Bq, R² = -2.446
A0 = 1.57e+05 ± 1.70e+04 Bq, R² = 0.999
A0 (single point) = 4.61e+04 ± 3.80e+04 Bq
A0 (single point) = 1.50e+05 ± 2.45e+04 Bq
A0 = 4.74e+04 ± 6.11e+03 Bq, R² = -0.055
A0 = 2.12e+05 ± 3.38e+04 Bq, R² = 0.937
